In [34]:
import pandas as pd
import numpy as np

In [35]:
# Sample size for exploration 
# (For this exploratory ntbook I'll assume the datasets might not be small to load in to jupyter ntbook)
SAMPLE_SIZE = 1000

df_survey = pd.read_csv('../data/survey_results.csv', nrows=SAMPLE_SIZE)
df_user = pd.read_csv('../data/user_metadata.csv', nrows=SAMPLE_SIZE)

In [36]:
# Type checking - Survey Results
print("\n=== Survey Results - Info ===")
df_survey.info()
print("\n=== Survey Results - Sample ===")
df_survey.head()


=== Survey Results - Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 809 entries, 0 to 808
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   submission_id  809 non-null    object 
 1   timestamp      809 non-null    object 
 2   user_email     809 non-null    object 
 3   rating         808 non-null    float64
 4   comment_text   809 non-null    object 
 5   region         809 non-null    object 
dtypes: float64(1), object(5)
memory usage: 38.0+ KB

=== Survey Results - Sample ===


,submission_id,timestamp,user_email,rating,comment_text,region
0,a8d2b6ad-a8b9-4621-92cf-177a8b0d2c73,2024-01-09T02:22:24,liam.smith@examplecorp.com,4.0,Fantastic.,APAC
1,a2ca6df0-59c0-4311-a281-12694ddee7d7,2024-03-21T05:07:44,noah.lopez@examplecorp.com,4.0,Fantastic.,APAC
2,226a1d8b-342f-46df-aacb-bfc6f4142d84,2024-01-12T18:14:20,ava.garcia@examplecorp.com,5.0,Fantastic.,Americas
3,21504601-ce62-4c48-ae9b-42275732a10c,2024-01-15T08:47:31,sophia.johnson26@examplecorp.com,4.0,Great experience.,Americas
4,190c4dee-af10-4e70-a35b-fe38db104021,2024-01-05T03:25:06,sophia.nielsen@examplecorp.com,5.0,Smooth process.,EMEA


Comments: timestamp not imported as datetime
___

In [37]:
# Type checking - User Metadata
print("\n=== User Metadata - Info ===")
df_user.info()
print("\n=== User Metadata - Sample ===")
df_user.head()


=== User Metadata - Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_email  62 non-null     object
 1   full_name   62 non-null     object
 2   department  62 non-null     object
 3   country     61 non-null     object
dtypes: object(4)
memory usage: 2.1+ KB

=== User Metadata - Sample ===


,user_email,full_name,department,country
0,olivia.nielsen@examplecorp.com,Olivia Nielsen,Marketing,Australia
1,liam.lopez@examplecorp.com,Liam Lopez,Customer Support,Germany
2,lucas.lopez@examplecorp.com,Lucas Lopez,Finance,Germany
3,sophia.garcia@examplecorp.com,Sophia Garcia,Product,United States
4,liam.garcia@examplecorp.com,Liam Garcia,Talent Acquisition,India


In [38]:
# Check for missing values
print("=== Missing Values - Survey Results ===")
print(df_survey.isnull().sum())
print("\n=== Missing Values - User Metadata ===")
print(df_user.isnull().sum())

=== Missing Values - Survey Results ===
submission_id    0
timestamp        0
user_email       0
rating           1
comment_text     0
region           0
dtype: int64

=== Missing Values - User Metadata ===
user_email    0
full_name     0
department    0
country       1
dtype: int64


Comments: we only see missing values in country and rating from this sample. We will assume that rating is a necessary field for analysis. For missing country values, we will fill with "Not specified". For missing ratings, we will drop those rows. Ratings will also be clipped to the valid range 1-5.

In [39]:
# Check for duplicates
print("=== Duplicate Check ===")
print(f"Survey duplicates by submission_id: {df_survey['submission_id'].duplicated().sum()}")
print(f"User duplicates by user_email: {df_user['user_email'].duplicated().sum()}")

=== Duplicate Check ===
Survey duplicates by submission_id: 1
User duplicates by user_email: 0


Comments: survey summisision_id seems like an automatically generated GUID. It might happen that emails too get duplicates (multiple survey submissions by the same user).
___

In [40]:
# Data sanitization - Convert types appropriately
# Convert timestamp to datetime, coerce errors to NaT (Not a Time)
df_survey['timestamp'] = pd.to_datetime(df_survey['timestamp'], errors='coerce')

# Detect and report invalid timestamps
invalid_timestamps = df_survey['timestamp'].isna().sum()
if invalid_timestamps > 0:
    print(f"{invalid_timestamps} invalid timestamp(s) found and set to NaT.")

# Strip whitespace from string columns
string_cols_survey = ['submission_id', 'user_email', 'comment_text', 'region', 'country']
for col in string_cols_survey:
    if col in df_survey.columns:
        df_survey[col] = df_survey[col].str.strip()

string_cols_user = ['user_email', 'full_name', 'department', 'country']
for col in string_cols_user:
    df_user[col] = df_user[col].str.strip()

# Normalize email to lowercase
df_survey['user_email'] = df_survey['user_email'].str.lower()
df_user['user_email'] = df_user['user_email'].str.lower()

# Fill missing country values with "Not specified"
if 'country' in df_survey.columns:
    df_survey['country'] = df_survey['country'].fillna('Not specified')

# Drop rows with missing rating values and clip rating to 1-5
if 'rating' in df_survey.columns:
    df_survey = df_survey.dropna(subset=['rating'])
    df_survey['rating'] = df_survey['rating'].clip(lower=1, upper=5)

print("Data sanitization complete!")

1 invalid timestamp(s) found and set to NaT.
Data sanitization complete!


Comments:
1. Convert invalid timestamps to NaT (not a time) to preserve column's datatype integrity.
2. Removed trailing white spaces. We will assume the name field is an open field where you can just put your name+surname or whatever. The important part for the user's validity is the email.
___

In [41]:
# Print duplicated submission_id rows for comparison
dupe_mask = df_survey['submission_id'].duplicated(keep=False)
dupe_rows = df_survey[dupe_mask]
if not dupe_rows.empty:
    print("Duplicated submission_id rows:")
    print(dupe_rows.sort_values('submission_id'))
else:
    print("No duplicated submission_id found.")

# Drop the first occurrence of each duplicate submission_id
dupe_ids = df_survey['submission_id'][df_survey['submission_id'].duplicated()].unique()
for dupe_id in dupe_ids:
    dupe_indices = df_survey.index[df_survey['submission_id'] == dupe_id].tolist()
    # Drop the first occurrence
    idx_to_drop = dupe_indices[0]
    print(f"Dropping first occurrence of duplicate: {dupe_id} at index {idx_to_drop}")
    df_survey = df_survey.drop(idx_to_drop)
df_survey = df_survey.reset_index(drop=True)

Duplicated submission_id rows:
                            submission_id           timestamp  \
0    a8d2b6ad-a8b9-4621-92cf-177a8b0d2c73 2024-01-09 02:22:24   
800  a8d2b6ad-a8b9-4621-92cf-177a8b0d2c73 2024-01-09 02:22:24   

                     user_email  rating comment_text region  
0    liam.smith@examplecorp.com     4.0   Fantastic.   APAC  
800  liam.smith@examplecorp.com     4.0   Fantastic.   APAC  
Dropping first occurrence of duplicate: a8d2b6ad-a8b9-4621-92cf-177a8b0d2c73 at index 0


Comments: it seems the submission_id is a random generated GUID that is susceptible to duplications. Ideally we want unique submission_ids, so we don't forward users to other user's surveys by mistake. On how to prevent that we'll talk more about in the system design schema.

In [42]:
# Type checking - Sanitized Survey Results
print("\n=== Sanitized Survey Results - Info ===")
df_survey.info()
print("\n=== Sanitized Survey Results - Sample ===")
df_survey.head()


=== Sanitized Survey Results - Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 807 entries, 0 to 806
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   submission_id  807 non-null    object        
 1   timestamp      806 non-null    datetime64[ns]
 2   user_email     807 non-null    object        
 3   rating         807 non-null    float64       
 4   comment_text   807 non-null    object        
 5   region         807 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(4)
memory usage: 38.0+ KB

=== Sanitized Survey Results - Sample ===


,submission_id,timestamp,user_email,rating,comment_text,region
0,a2ca6df0-59c0-4311-a281-12694ddee7d7,2024-03-21 05:07:44,noah.lopez@examplecorp.com,4.0,Fantastic.,APAC
1,226a1d8b-342f-46df-aacb-bfc6f4142d84,2024-01-12 18:14:20,ava.garcia@examplecorp.com,5.0,Fantastic.,Americas
2,21504601-ce62-4c48-ae9b-42275732a10c,2024-01-15 08:47:31,sophia.johnson26@examplecorp.com,4.0,Great experience.,Americas
3,190c4dee-af10-4e70-a35b-fe38db104021,2024-01-05 03:25:06,sophia.nielsen@examplecorp.com,5.0,Smooth process.,EMEA
4,bd501dc1-dfed-4574-bd84-69253db2d46a,2024-01-16 19:17:21,olivia.johnson@examplecorp.com,5.0,Fantastic.,Americas


In [43]:
# Type checking - Sanitized User Metadata
print("\n=== Sanitized User Metadata - Info ===")
df_user.info()
print("\n=== Sanitized User Metadata - Sample ===")
df_user.head()


=== Sanitized User Metadata - Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_email  62 non-null     object
 1   full_name   62 non-null     object
 2   department  62 non-null     object
 3   country     61 non-null     object
dtypes: object(4)
memory usage: 2.1+ KB

=== Sanitized User Metadata - Sample ===


,user_email,full_name,department,country
0,olivia.nielsen@examplecorp.com,Olivia Nielsen,Marketing,Australia
1,liam.lopez@examplecorp.com,Liam Lopez,Customer Support,Germany
2,lucas.lopez@examplecorp.com,Lucas Lopez,Finance,Germany
3,sophia.garcia@examplecorp.com,Sophia Garcia,Product,United States
4,liam.garcia@examplecorp.com,Liam Garcia,Talent Acquisition,India


In [44]:
# Validate data ranges and categorical values
print("=== Value Validation ===")
print(f"\nRating range: {df_survey['rating'].min()} - {df_survey['rating'].max()}")
print(f"\nUnique regions: {df_survey['region'].unique()}")
print(f"\nUnique departments: {df_user['department'].unique()}")
print(f"\nUnique countries: {df_user['country'].unique()}")
# Validate no duplicated submission_id after sanitization
dupe_count = df_survey['submission_id'].duplicated().sum()
if dupe_count > 0:
    print(f"\nWarning: {dupe_count} duplicated submission_id(s) remain after sanitization!")
else:
    print("\nAll submission_id values are unique after sanitization.")

=== Value Validation ===

Rating range: 1.0 - 5.0

Unique regions: ['APAC' 'Americas' 'EMEA']

Unique departments: ['Marketing' 'Customer Support' 'Finance' 'Product' 'Talent Acquisition'
 'Engineering' 'Sales' 'HR']

Unique countries: ['Australia' 'Germany' 'United States' 'India' 'Denmark' 'Canada'
 'United Kingdom' nan]

All submission_id values are unique after sanitization.
